# 학습한 모델을 서비스로 배포하기

`dm_train.ipynb`에서 만든 모델(`dm_model_best.pt`)을 **어디서든 호출 가능한 API**로 만듭니다.

절차: 모델 변환(TorchScript) → 배포 패키지 생성 → 포털 Serving 등록 → API 호출

> 먼저 `dm_train.ipynb`의 6번 셀까지 실행해 `dm_model_best.pt`가 있어야 합니다.

## 1. 학습한 모델 불러오기

In [ ]:
import torch, torch.nn as nn
from torchvision.models import resnet18

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, 2)
model.load_state_dict(torch.load("dm_model_best.pt", map_location=device))
model.eval().to(device)
print("학습된 모델 로드 완료")

## 2. 배포용으로 변환 (TorchScript)

파이썬 코드 없이도 실행 가능한 형태로 바꿉니다.
서빙 서버는 이 파일 하나만 있으면 추론할 수 있습니다.

In [ ]:
import os

example = torch.randn(1, 3, 128, 128).to(device)
traced = torch.jit.trace(model, example)

os.makedirs("deploy/dm-classifier/1", exist_ok=True)
traced.save("deploy/dm-classifier/1/model.pt")

size_mb = os.path.getsize("deploy/dm-classifier/1/model.pt") / 1024 / 1024
print(f"변환 완료: deploy/dm-classifier/1/model.pt ({size_mb:.1f} MB)")

## 3. 변환된 모델 검증

변환 전후 결과가 같은지 확인합니다. 같아야 배포해도 안전합니다.

In [ ]:
from PIL import Image
from torchvision import transforms
from pathlib import Path

tf = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

sample = sorted(Path("qr_crops_failed").glob("*.jpg"))[0]
x = tf(Image.open(sample).convert("RGB")).unsqueeze(0).to(device)

with torch.no_grad():
    a = torch.softmax(model(x), 1)[0]
    b = torch.softmax(traced(x), 1)[0]

print("원본 모델 :", [f"{v:.4f}" for v in a.tolist()])
print("변환 모델 :", [f"{v:.4f}" for v in b.tolist()])
print("결과 일치 :", torch.allclose(a, b, atol=1e-4))
print()
print("판정:", ["판독 성공 예상", "판독 실패 예상"][int(b.argmax())],
      f"(확신도 {b.max()*100:.1f}%)")

## 4. 서빙용 설정 파일 만들기

서버가 이미지를 어떻게 전처리할지 알려주는 정보를 함께 저장합니다.

In [ ]:
import json

meta = {
    "name": "dm-classifier",
    "task": "DataMatrix 판독 가능성 분류",
    "classes": ["OK(판독 성공)", "FAIL(판독 실패)"],
    "input": {"size": [128, 128], "channels": 3,
              "normalize": {"mean": [0.485, 0.456, 0.406],
                            "std": [0.229, 0.224, 0.225]}},
    "trained_on": {"ok": 3636, "fail": 2015},
    "val_accuracy": 0.9496,
}
with open("deploy/dm-classifier/config.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print(json.dumps(meta, ensure_ascii=False, indent=2))

## 5. 배포 패키지 압축

포털에서 등록할 수 있도록 하나의 파일로 묶습니다.

In [ ]:
import shutil

shutil.make_archive("dm-classifier", "zip", "deploy")
mb = os.path.getsize("dm-classifier.zip") / 1024 / 1024
print(f"배포 패키지 생성: dm-classifier.zip ({mb:.1f} MB)")
print()
print("다음 단계:")
print(" 1) 왼쪽 파일 목록에서 dm-classifier.zip 우클릭 → Download")
print(" 2) 담당자에게 전달하거나, 포털 Serving 탭에서 등록")

## 6. 로컬에서 API처럼 호출해보기

배포 전에, 서빙 서버가 하는 일을 그대로 흉내내어 테스트합니다.

In [ ]:
import random
from pathlib import Path

def predict(image_path):
    """이미지 경로를 주면 판정 결과를 돌려준다 (서빙 API와 동일한 동작)"""
    img = Image.open(image_path).convert("RGB")
    x = tf(img).unsqueeze(0).to(device)
    with torch.no_grad():
        p = torch.softmax(traced(x), 1)[0]
    idx = int(p.argmax())
    return {"판정": ["OK", "FAIL"][idx],
            "확신도": round(float(p[idx]) * 100, 1),
            "OK확률": round(float(p[0]) * 100, 1),
            "FAIL확률": round(float(p[1]) * 100, 1)}

tests = (random.sample(sorted(Path("qr_crops").glob("*.jpg")), 3)
         + random.sample(sorted(Path("qr_crops_failed").glob("*.jpg")), 3))

for f in tests:
    truth = "OK" if "failed" not in str(f.parent) else "FAIL"
    r = predict(f)
    mark = "O" if r["판정"] == truth else "X"
    print(f"[{mark}] {f.name[:35]:35} 정답={truth:4} AI={r['판정']:4} ({r['확신도']}%)")

## 7. 정리

- **만든 것**: DM코드 판독 가능성을 95% 정확도로 판정하는 모델
- **배포 파일**: `dm-classifier.zip` (모델 + 설정)
- **활용**: 촬영 즉시 재촬영 필요 여부를 알려주는 현장 검사 도구

### 실제 서비스로 올리려면
1. 사내 모델 저장소(MinIO/S3)에 `dm-classifier` 폴더 업로드
2. 포털 → AI 개발환경 → Serving → 모델 등록 (경로 지정)
3. 배포되면 `POST /v1/models/dm-classifier:predict` 로 호출 가능

> 현재 개발 환경에서는 모델 저장소가 임시 구성이라, 배포는 담당자와 협의 후 진행하세요.